# Visualize Point Labeler Export Masks

Use this notebook to inspect masks exported from `point_labeler/scripts/export_from_point_labeler.py`, for example `export/000000/semantic_mask.npy`. For flat-point exports, the mask is a 1D array where `mask[i]` corresponds to `velodyne/<frame>.bin` point `i` and CSV row `i`.

In [ ]:
from __future__ import annotations

import json
import math
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

plt.rcParams["figure.figsize"] = (16, 7)
plt.rcParams["axes.grid"] = False

LABELER_DIR = Path("/home/a60116606/git_repo/point_labeler/HL320_output_sam3_manual_104")
EXPORT_DIR = LABELER_DIR / "export"

# Change this to inspect another exported frame.
FRAME_ID = "000000"
MASK_PATH = EXPORT_DIR / FRAME_ID / "semantic_mask.npy"

print("LABELER_DIR:", LABELER_DIR)
print("EXPORT_DIR:", EXPORT_DIR)
print("MASK_PATH:", MASK_PATH)


In [ ]:
def read_json(path: Path) -> dict:
    if not path.is_file():
        return {}
    return json.loads(path.read_text(encoding="utf-8"))


def read_labels_xml(path: Path) -> tuple[dict[int, str], dict[int, np.ndarray]]:
    if not path.is_file():
        return {}, {}
    root = ET.parse(path).getroot()
    id_to_name = {}
    id_to_color = {}
    for label in root.findall("label"):
        name = (label.findtext("name") or "").strip()
        raw_id = (label.findtext("id") or "").strip()
        if not name or not raw_id:
            continue
        label_id = int(raw_id)
        id_to_name[label_id] = name
        raw_color = (label.findtext("color") or "").strip()
        if raw_color:
            values = [int(float(token)) for token in raw_color.replace(",", " ").split()[:3]]
            if len(values) == 3:
                id_to_color[label_id] = np.asarray(values, dtype=np.uint8)
    return id_to_name, id_to_color


bridge_manifest = read_json(LABELER_DIR / "bridge_manifest.json")
frame_manifest_by_id = {str(frame.get("frame_id")): frame for frame in bridge_manifest.get("frames", [])}
id_to_name, id_to_color = read_labels_xml(LABELER_DIR / "labels.xml")
id_to_name.setdefault(255, "ignore")
id_to_color.setdefault(255, np.array([125, 125, 125], dtype=np.uint8))
print("manifest label_layout:", bridge_manifest.get("label_layout"))
print("frames in manifest:", len(frame_manifest_by_id))
print("classes:", len(id_to_name))


In [ ]:
def split_table_row(line: str) -> list[str]:
    if "," in line:
        return [value.strip() for value in line.split(",")]
    return line.replace("\t", " ").split()


def frame_paths(frame_id: str) -> dict[str, Path | None]:
    frame = frame_manifest_by_id.get(frame_id, {})
    csv_path = Path(frame["source_csv"]) if frame.get("source_csv") else None
    if csv_path is not None and not csv_path.is_file():
        csv_path = None
    if csv_path is None:
        for folder in ("csv", "CSV"):
            candidate = LABELER_DIR / folder / f"{frame_id}.csv"
            if candidate.is_file():
                csv_path = candidate
                break

    image_path = None
    for key in ("image", "source_image"):
        raw = frame.get(key)
        if raw and Path(raw).is_file():
            image_path = Path(raw)
            break
    if image_path is None:
        for suffix in (".jpg", ".jpeg", ".png", ".bmp", ".webp"):
            candidate = LABELER_DIR / "image_2" / f"{frame_id}{suffix}"
            if candidate.is_file():
                image_path = candidate
                break

    return {
        "mask": EXPORT_DIR / frame_id / "semantic_mask.npy",
        "velodyne": LABELER_DIR / "velodyne" / f"{frame_id}.bin",
        "csv": csv_path,
        "image": image_path,
    }


def load_points(path: Path) -> np.ndarray | None:
    if not path.is_file():
        return None
    raw = np.fromfile(path, dtype=np.float32)
    if raw.size % 4 != 0:
        raise ValueError(f"{path} is not float32 XYZI")
    return raw.reshape(-1, 4)


def load_csv_columns(path: Path | None) -> dict[str, np.ndarray]:
    if path is None or not path.is_file():
        return {}
    with path.open("r", encoding="utf-8") as handle:
        header = None
        for line in handle:
            if line.strip():
                header = split_table_row(line.strip())
                break
        if header is None:
            return {}
        column_map = {name.strip().casefold(): index for index, name in enumerate(header)}
        wanted = ["cxd", "cyd", "azimuth", "vertical", "slot", "pixel", "intensity"]
        values = {name: [] for name in wanted if name in column_map}
        max_idx = max(column_map[name] for name in values) if values else -1
        for line in handle:
            if not line.strip():
                continue
            tokens = split_table_row(line.strip())
            if len(tokens) <= max_idx:
                continue
            for name in values:
                try:
                    values[name].append(float(tokens[column_map[name]]))
                except ValueError:
                    values[name].append(np.nan)
    return {name: np.asarray(items, dtype=np.float64) for name, items in values.items()}


def load_image(path: Path | None) -> np.ndarray | None:
    if path is None or not path.is_file():
        return None
    from PIL import Image
    return np.asarray(Image.open(path).convert("RGB"), dtype=np.uint8)


In [ ]:
def class_name(label_id: int) -> str:
    return id_to_name.get(int(label_id), f"id_{int(label_id)}")


def stable_color(label_id: int) -> np.ndarray:
    label_id = int(label_id)
    if label_id in id_to_color:
        return id_to_color[label_id]
    if label_id == 0:
        return np.array([35, 35, 35], dtype=np.uint8)
    if label_id == 255:
        return np.array([125, 125, 125], dtype=np.uint8)
    rng = np.random.default_rng((label_id * 1009 + 17) & 0xFFFFFFFF)
    return rng.integers(40, 240, size=3, dtype=np.uint8)


def colors_for_labels(labels: np.ndarray) -> np.ndarray:
    flat = np.asarray(labels).reshape(-1)
    colors = np.zeros((flat.size, 3), dtype=np.uint8)
    for label_id in np.unique(flat):
        colors[flat == label_id] = stable_color(int(label_id))
    return colors


def print_distribution(labels: np.ndarray, max_rows: int = 60) -> None:
    flat = np.asarray(labels).reshape(-1)
    ids, counts = np.unique(flat, return_counts=True)
    pairs = sorted([(int(i), int(c)) for i, c in zip(ids, counts)], key=lambda item: item[1], reverse=True)
    rows = ["| class id | name | points | percent |", "|---:|---|---:|---:|"]
    for label_id, count in pairs[:max_rows]:
        rows.append(f"| {label_id} | {class_name(label_id)} | {count} | {100.0 * count / flat.size:.3f}% |")
    display(Markdown("\n".join(rows)))


def sample_indices(n: int, max_points: int | None = 300_000) -> np.ndarray:
    idx = np.arange(n)
    if max_points is not None and n > max_points:
        rng = np.random.default_rng(42)
        idx = np.sort(rng.choice(idx, size=max_points, replace=False))
    return idx


In [ ]:
def load_export_frame(frame_id: str):
    paths = frame_paths(frame_id)
    mask = np.load(paths["mask"], allow_pickle=False).reshape(-1)
    points = load_points(paths["velodyne"])
    csv_cols = load_csv_columns(paths["csv"])
    image = load_image(paths["image"])
    return mask, points, csv_cols, image, paths


mask, points, csv_cols, image, paths = load_export_frame(FRAME_ID)
print("frame:", FRAME_ID)
print("mask:", paths["mask"], mask.shape, mask.dtype)
print("velodyne:", paths["velodyne"], None if points is None else points.shape)
print("csv:", paths["csv"], {k: v.shape for k, v in csv_cols.items()})
print("image:", paths["image"], None if image is None else image.shape)
if points is not None:
    print("mask/points same count:", mask.size == points.shape[0])
if csv_cols:
    first_len = next(iter(csv_cols.values())).size
    print("mask/csv same count:", mask.size == first_len)
print_distribution(mask)


In [ ]:
def visualize_export_frame(frame_id: str, *, max_points: int | None = 300_000, point_size: float = 0.8) -> None:
    labels, points, csv_cols, image, paths = load_export_frame(frame_id)
    idx = sample_indices(labels.size, max_points=max_points)
    colors = colors_for_labels(labels[idx]).astype(np.float32) / 255.0

    fig, axes = plt.subplots(2, 2, figsize=(22, 14))
    axes = axes.reshape(-1)

    if points is not None and points.shape[0] == labels.size:
        pts = points[idx]
        axes[0].scatter(pts[:, 0], pts[:, 1], c=colors, s=point_size, linewidths=0)
        axes[0].set_title("LiDAR XY top view")
        axes[0].set_xlabel("x")
        axes[0].set_ylabel("y")
        axes[0].axis("equal")

        axes[1].scatter(pts[:, 0], pts[:, 2], c=colors, s=point_size, linewidths=0)
        axes[1].set_title("LiDAR XZ side view")
        axes[1].set_xlabel("x")
        axes[1].set_ylabel("z")
        axes[1].axis("equal")
    else:
        axes[0].text(0.5, 0.5, "velodyne missing or count mismatch", ha="center", va="center")
        axes[1].text(0.5, 0.5, "velodyne missing or count mismatch", ha="center", va="center")

    if {"cxd", "cyd"}.issubset(csv_cols) and csv_cols["cxd"].size == labels.size:
        cxd = csv_cols["cxd"][idx]
        cyd = csv_cols["cyd"][idx]
        finite = np.isfinite(cxd) & np.isfinite(cyd)
        if image is not None:
            axes[2].imshow(image)
            axes[2].set_xlim(0, image.shape[1])
            axes[2].set_ylim(image.shape[0], 0)
        else:
            axes[2].invert_yaxis()
        axes[2].scatter(cxd[finite], cyd[finite], c=colors[finite], s=point_size, linewidths=0, alpha=0.85)
        axes[2].set_title("Camera projection via CSV Cxd/Cyd")
        axes[2].set_xlabel("Cxd")
        axes[2].set_ylabel("Cyd")
    else:
        axes[2].text(0.5, 0.5, "CSV Cxd/Cyd missing or count mismatch", ha="center", va="center")

    if {"azimuth", "vertical"}.issubset(csv_cols) and csv_cols["azimuth"].size == labels.size:
        x = -1.0 * csv_cols["azimuth"][idx]
        y = -1.0 * csv_cols["vertical"][idx]
        title = "Angular view: azimuth / vertical"
        xlabel, ylabel = "-azimuth", "-vertical"
    elif {"pixel", "slot"}.issubset(csv_cols) and csv_cols["pixel"].size == labels.size:
        x = csv_cols["pixel"][idx]
        y = csv_cols["slot"][idx]
        title = "Acquisition view: pixel / slot"
        xlabel, ylabel = "pixel", "slot"
    else:
        x = y = None
    if x is not None:
        finite = np.isfinite(x) & np.isfinite(y)
        axes[3].scatter(x[finite], y[finite], c=colors[finite], s=point_size, linewidths=0)
        axes[3].set_title(title)
        axes[3].set_xlabel(xlabel)
        axes[3].set_ylabel(ylabel)
        axes[3].invert_yaxis()
    else:
        axes[3].text(0.5, 0.5, "No angular/acquisition CSV columns", ha="center", va="center")

    plt.suptitle(f"Exported point_labeler mask: {frame_id}, {labels.size} points", y=1.02)
    plt.tight_layout()
    plt.show()


visualize_export_frame(FRAME_ID)


In [ ]:
# Browse several exported frames.
frame_ids = sorted(path.name for path in EXPORT_DIR.iterdir() if path.is_dir())
print("exported frames:", len(frame_ids), frame_ids[:10])

start = 0
count = 5
for frame_id in frame_ids[start : start + count]:
    visualize_export_frame(frame_id, max_points=150_000, point_size=0.45)


In [ ]:
def save_camera_projection_png(frame_id: str, out_dir: Path | None = None, *, point_size: float = 0.7) -> Path:
    labels, points, csv_cols, image, paths = load_export_frame(frame_id)
    if not {"cxd", "cyd"}.issubset(csv_cols) or csv_cols["cxd"].size != labels.size:
        raise ValueError(f"No matching Cxd/Cyd CSV data for {frame_id}")
    colors = colors_for_labels(labels).astype(np.float32) / 255.0
    cxd = csv_cols["cxd"]
    cyd = csv_cols["cyd"]
    finite = np.isfinite(cxd) & np.isfinite(cyd)
    fig, ax = plt.subplots(1, 1, figsize=(14, 9))
    if image is not None:
        ax.imshow(image)
        ax.set_xlim(0, image.shape[1])
        ax.set_ylim(image.shape[0], 0)
    else:
        ax.invert_yaxis()
    ax.scatter(cxd[finite], cyd[finite], c=colors[finite], s=point_size, linewidths=0, alpha=0.85)
    ax.set_title(f"{frame_id} exported mask projected to camera")
    ax.axis("off")
    out_dir = out_dir or (EXPORT_DIR / "camera_previews")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{frame_id}_export_camera_projection.png"
    fig.savefig(out_path, dpi=180, bbox_inches="tight")
    plt.close(fig)
    return out_path


# Uncomment to save all camera projection previews.
# saved = [save_camera_projection_png(frame_id) for frame_id in frame_ids]
# print(f"saved {len(saved)} previews to {saved[0].parent if saved else ''}")
